In [ ]:
pip install "transformers[torch]"

In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [ ]:
val_data=pd.read_csv("samsum-validation.csv")

In [ ]:
train_data=pd.read_csv("samsum-train.csv")



In [ ]:
train_data.head()

In [ ]:
train_data.shape

In [ ]:
val_data.shape

In [ ]:
train_data["dialogue"][0]

In [ ]:
train_data=train_data.sample(n=5000,random_state=42).reset_index(drop=True)
val_data=val_data.sample(n=500,random_state=42).reset_index(drop=True)

Data Pre -Processing

In [ ]:
import re
def clean_data(text):
  text=re.sub(r"\r\n"," ",text) #lines
  text=re.sub(r"\s+"," ",text)#spaces
  text=re.sub(r"<.*?>"," ",text)#html tags <p> <h1>
  text=text.strip() #no lower() because t5 is a cased model
  return text

In [ ]:
train_data["dialogues"]=train_data['dialogue'].apply(clean_data)
val_data["dialogues"]=val_data['dialogue'].apply(clean_data)
train_data["summary"]=train_data['summary'].apply(clean_data)
val_data["summary"]=val_data['summary'].apply(clean_data)

In [ ]:
train_data["dialogues"][0]

## **#Tokenize**

In [ ]:
tokenizer=T5Tokenizer.from_pretrained("google/flan-t5-base")

In [ ]:
#raw data convert to tokenized inputs  for fine-tuning
def tokenize(data):
  inputs=tokenizer("summarize: "+data["dialogues"],padding="max_length",max_length=512,truncation=True)
  target=tokenizer(data["summary"],padding="max_length",max_length=150,truncation=True)
  inputs["labels"]=[t if t!=tokenizer.pad_token_id else -100 for t in target["input_ids"]] #token ids is added to input as labels, padding is replaced by -100 so loss ignores it
  return inputs

In [ ]:
train_dataset=train_data.apply(tokenize,axis=1).tolist()
val_dataset=val_data.apply(tokenize,axis=1).tolist()

In [ ]:
print(train_dataset[0])

In [ ]:
#input_ids-dialogues tokens ids
#1= end of sequence and 0 represents padding
#attention masks-
#labels-summary tokens ids
#-100 in labels=padding, loss ignores it


In [ ]:
len(train_dataset[0]['input_ids'])

In [ ]:
len(train_dataset[0]['labels'])

# **WORKING WITH OUR MODEL**

In [ ]:
#It is nlp task we are generating summary which is text

#but this genration ofsummary is conditional we are giving dialogue and on the basis of this it is generating summary
#it is dependent on it for the generation that is why this model


model=T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")


In [ ]:
# fine tune
import torch
if torch.backends.mps.is_available():
  device=torch.device("mps")
elif torch.cuda.is_available():
  device=torch.device("cuda")
else:
  device=torch.device("cpu")
print("device:",device)
model.to(device)

In [ ]:
# training arguments we need to define
training_args=TrainingArguments(
    output_dir='./results',

    num_train_epochs=4,
    learning_rate=1e-4,
    weight_decay=0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy="epoch",
    save_strategy="epoch",

    warmup_steps=500,

    load_best_model_at_end=True, #keeps the best epoch not the last
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2
)

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

TRAINING THE MODEL

In [ ]:
trainer.train()

# **SAVE THE MODEL**

In [ ]:
model.save_pretrained("./saved_model_summary")
tokenizer.save_pretrained("./saved_model_summary")

In [ ]:
model= T5ForConditionalGeneration.from_pretrained("./saved_model_summary")
tokenizer=T5Tokenizer.from_pretrained("./saved_model_summary")

# **TEST THE CORE LOGIC OF SUMMARIZATION**

In [ ]:
def summarize_dialogue(dialogue):
  dialogue = clean_data(dialogue) #preprocessing
  dialogue = "summarize: " + dialogue #same prefix as training

  #tokenizing with return_tensors="pt" for PyTorch
  inputs = tokenizer(
      dialogue,
      padding="max_length",
      max_length=512,
      truncation=True,
      return_tensors="pt"
  ).to(device)
  model.to(device)

  #generating the summary => token ids
  targets = model.generate(
      input_ids=inputs["input_ids"],
      attention_mask=inputs["attention_mask"],
      max_length=150,
      num_beams=4,
      no_repeat_ngram_size=3,
      early_stopping=True
  )

  #token ids convert to text or summary => decoding
  summary = tokenizer.decode(
      targets[0],
      skip_special_tokens=True
  )
  return summary

In [ ]:
test_dialogue="""Alex Chen, Senior Tech Correspondent (The Daily Chronicle): Dr. Thorne, thank you for joining us. We’ve spent the last three years watching generative AI transition from a novelty to a boardroom mandate. As we sit here today, what is the most consequential shift happening in artificial intelligence that the general public is missing?

Dr. Elena Thorne, Director of the Institute for Applied AI Ethics: Thanks for having me, Alex. The public is still largely focused on the conversational aspect—chatbots writing emails or generating images. But the real earthquake is the shift from reactive AI to agentic AI. We are no longer just prompting systems for answers; we are deploying autonomous agents that can execute multi-step workflows. An AI isn't just drafting a quarterly report anymore; it's pulling the raw data from a secure server, analyzing the financial discrepancies, drafting the report, and emailing it to the compliance team without human intervention. That leap in autonomy is where the massive productivity gains lie, but it’s also where the systemic risks multiply.

Alex Chen: Let’s talk about those risks. We are seeing a very fragmented approach to regulation globally. The EU has its AI Act, the US is relying heavily on a patchwork of state laws and executive orders, and regions like Asia-Pacific are rapidly accelerating their own frameworks. How does a multinational company even navigate this?

Dr. Elena Thorne: With extreme difficulty. What we are seeing in 2026 is a massive "compliance gap." Regulators are increasingly realizing that governing the model is nearly impossible because the technology moves too fast. Instead, they are aggressively targeting data governance. The common thread across jurisdictions—whether it's the EU, California, or India's new digital frameworks—is that you must control and document the data your AI touches. The biggest threat to enterprises right now isn't a rogue superintelligence; it’s "shadow AI." It’s the employee pasting proprietary source code or sensitive client data into an unapproved, consumer-grade AI tool to save thirty minutes of work.

Alex Chen: That sounds like a nightmare for Chief Information Security Officers. But let's pivot to the human element. The anxiety around job displacement hasn't gone away. Are we finally hitting the point where AI is replacing white-collar workers at scale?

Dr. Elena Thorne: The narrative of a sudden, mass extinction of jobs is overblown, but the reality of a "skills squeeze" is very real. We are seeing a hollowing out of entry-level cognitive work. The tasks we used to give to junior analysts, paralegals, or associate developers—summarizing documents, basic code debugging, preliminary research—are being fully automated. This creates a pipeline problem. How do you train a senior strategist if the junior roles they traditionally cut their teeth on no longer exist? Companies have to fundamentally restructure their apprenticeship models. The future workforce isn't humans versus AI; it’s humans collaborating with AI versus humans who refuse to adapt.

Alex Chen: So the premium is now on strategic thinking and managing the AI's output rather than generating the raw material.

Dr. Elena Thorne: Exactly. The human becomes the editor and the auditor, not the blank-page creator.

Alex Chen: To wrap up, if you had to identify one aspect of AI that is currently overhyped, and one that is dangerously underhyped, what would they be?

Dr. Elena Thorne: The most overhyped concept is the imminent arrival of Artificial General Intelligence, or AGI. We are still fundamentally dealing with sophisticated pattern-matching algorithms, not sentient beings, and the billions of dollars poured into the AGI race often distract from immediate, practical problems.

The most underhyped aspect is AI's compounding impact on the physical sciences. We obsess over AI writing poetry, but AI models are currently predicting millions of new crystal structures for better batteries and simulating protein folding for targeted cancer therapies in a matter of days. That intersection of AI and material science is going to redefine our physical world over the next decade, and it gets a fraction of the media coverage.

Alex Chen: A fascinating reality check. Dr. Thorne, thank you for your time."""



print(summarize_dialogue(test_dialogue))

# **INTENT OF THE DIALOGUE**

In [ ]:
#our fine tuned model only gives summary (what happened)
#for intent (why it was said) we use base flan-t5 model because it can follow instructions

intent_tokenizer=T5Tokenizer.from_pretrained("google/flan-t5-base")
intent_model=T5ForConditionalGeneration.from_pretrained("google/flan-t5-base")
intent_model.to(device)

def detect_intent(dialogue):
  dialogue = clean_data(dialogue) #preprocessing

  #prompt tells the model what we want
  prompt = f"Conversation:\n{dialogue}\n\nWhat is the main intent or purpose of this conversation? Answer in one short sentence."

  inputs = intent_tokenizer(
      prompt,
      max_length=512,
      truncation=True,
      return_tensors="pt"
  ).to(device)

  #generating the intent => token ids
  targets = intent_model.generate(
      **inputs,
      max_new_tokens=40,
      num_beams=4,
      early_stopping=True
  )

  #token ids convert to text => decoding
  intent = intent_tokenizer.decode(
      targets[0],
      skip_special_tokens=True
  )
  return intent

In [ ]:
print(detect_intent(test_dialogue))

In [ ]:
import shutil
from google.colab import files

# Zip the folder (replace with your folder path)
shutil.make_archive("saved_model_summary", "zip", "/content/saved_model_summary")

# Download to your device /content/saved_model_summary
files.download("saved_model_summary.zip")